In [ ]:
!nvidia-smi # make sure we are using nvidia GPU
!pip install ultralytics roboflow pycocotools -q # allows us to use YOLOv11 with roboflow

Thu Jul 16 17:28:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!unzip "braille detection.v1i.coco.zip" -d braille_dataset
# unzips the directory with all of the braille images and puts it in a folder called "braille_dataset"

Archive:  braille detection.v1i.coco.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of braille detection.v1i.coco.zip or
        braille detection.v1i.coco.zip.zip, and cannot find braille detection.v1i.coco.zip.ZIP, period.


In [ ]:
!ls -lh "/content/braille detection.v1i.coco.zip"

-rw-r--r-- 1 root root 238M Jul 16 17:30 '/content/braille detection.v1i.coco.zip'


In [ ]:
!ls -lh "/content/braille detection.v1i.coco.zip"

-rw-r--r-- 1 root root 266M Jul 16 17:30 '/content/braille detection.v1i.coco.zip'


In [ ]:
import zipfile

zip_path = "/content/braille detection.v1i.coco.zip"

print("Is ZIP:", zipfile.is_zipfile(zip_path))

try:
    with zipfile.ZipFile(zip_path, "r") as z:
        print("First bad file:", z.testzip())
        print("Files:", len(z.namelist()))
except Exception as e:
    print("ZIP error:", e)

Is ZIP: True
First bad file: None
Files: 3896


In [ ]:
import zipfile
import os

zip_path = "/content/braille detection.v1i.coco.zip"
extract_path = "/content/braille_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction complete.")
print(os.listdir(extract_path)[:10])

Extraction complete.
['README.dataset.txt', 'valid', 'test', 'train', 'README.roboflow.txt']


## Unpacking Google Drive Mount Cell


In [ ]:
import torch
import os
import shutil

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

for file in os.listdir("/content/drive/MyDrive"):
    if "checkpoint" in file.lower() or "resnet" in file.lower() or "yolo" in file.lower():
        print(file)

braille_resnet_checkpoint_epoch_19.pth
braille_yolo_best.pt
braille_yolo_last.pt
resnet50_checkpoint_epoch_30.pth


Original Notebook Parts Below

In [ ]:
# checking what's in the directory
import os
print(os.listdir("braille_dataset"))

['README.dataset.txt', 'valid', 'test', 'train', 'README.roboflow.txt']


In [ ]:
# checking we are using the correct 4,000 image dataset

import os

for split in ["train", "valid", "test"]:
  folder = os.path.join("braille_dataset", split)
  num_images = len([f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
  print(f"{split}: {num_images} images")

# the total comes out to 3,891 so it is correct

train: 1823 images
valid: 974 images
test: 1094 images


In [ ]:
# checking the COCO annotation files

import os

for root, dirs, files in os.walk("braille_dataset"):
  json_files = [f for f in files if f.endswith(".json")]
  if json_files:
    print(root, json_files)


braille_dataset/valid ['_annotations.coco.json']
braille_dataset/test ['_annotations.coco.json']
braille_dataset/train ['_annotations.coco.json']


In [ ]:
# install training libraries

!pip install -q ultralytics pycocotools

In [ ]:
# quick check to make sure that it works

import ultralytics
print("Ultralytics installed")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics installed


In [ ]:
# convert COCO annotations to YOLO label format

from ultralytics.data.converter import convert_coco
from pathlib import Path

dataset_path = Path("braille_dataset")

convert_coco(
    labels_dir=str(dataset_path),
    save_dir="braille_yolo",
    use_segments=False,
    use_keypoints=False
)

COCO data converted successfully.
Results saved to /content/braille_yolo


In [ ]:
# check the converted folder

import os

for root, dirs, files in os.walk("braille_yolo"):
  if files[:5]:
    print(root, files[:5])

# checking exact split folders to make sure we have the right directory structure

import os

for root, dirs, files in os.walk("braille_yolo"):
    print(root, "dirs:", dirs, "num files:", len(files))

braille_yolo dirs: ['images', 'labels'] num files: 0
braille_yolo/images dirs: [] num files: 0
braille_yolo/labels dirs: [] num files: 0


In [ ]:
# error the first time around converting images --> this one accurately
# fixes those annotations and puts them into each folder (train, validate, test)

import json, os, shutil
from pathlib import Path
from PIL import Image

src = Path("braille_dataset")
out = Path("braille_yolo")

# reset output folder
if out.exists():
    shutil.rmtree(out)

for split in ["train", "valid", "test"]:
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    ann_path = src / split / "_annotations.coco.json"

    with open(ann_path, "r") as f:
        coco = json.load(f)

    # image id -> image info
    images = {img["id"]: img for img in coco["images"]}

    # collect annotations by image
    anns_by_img = {}
    for ann in coco["annotations"]:
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    for img_id, img in images.items():
        filename = img["file_name"]

        old_img_path = src / split / filename
        new_img_path = out / "images" / split / filename

        shutil.copy(old_img_path, new_img_path)

        width = img["width"]
        height = img["height"]

        label_path = out / "labels" / split / (Path(filename).stem + ".txt")

        with open(label_path, "w") as label_file:
            for ann in anns_by_img.get(img_id, []):
                x, y, w, h = ann["bbox"]

                x_center = (x + w / 2) / width
                y_center = (y + h / 2) / height
                w_norm = w / width
                h_norm = h / height

                class_id = ann["category_id"] - 1

                label_file.write(
                    f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n"
                )

print("Done converting COCO to YOLO format.")

Done converting COCO to YOLO format.


In [ ]:
# final check for real file counts

for root, dirs, files in os.walk("braille_yolo"):
    print(root, "dirs:", dirs, "num files:", len(files))

braille_yolo dirs: ['images', 'labels'] num files: 0
braille_yolo/images dirs: ['valid', 'test', 'train'] num files: 0
braille_yolo/images/valid dirs: [] num files: 974
braille_yolo/images/test dirs: [] num files: 1094
braille_yolo/images/train dirs: [] num files: 1823
braille_yolo/labels dirs: ['valid', 'test', 'train'] num files: 0
braille_yolo/labels/valid dirs: [] num files: 974
braille_yolo/labels/test dirs: [] num files: 1094
braille_yolo/labels/train dirs: [] num files: 1823


## Train YOLOv11 for 20 epochs and then get the **mAP score** from the test set

In [ ]:
from pathlib import Path

# path tells where the YOLO converted dataset is
# train, val, and test tell YOLO where the images are
# names tell YOLO that class 0 is A, class 1 is B, etc.

yaml_text = """
path: /content/braille_yolo
train: images/train
val: images/valid
test: images/test

names:
  0: A
  1: B
  2: C
  3: D
  4: E
  5: F
  6: G
  7: H
  8: I
  9: J
  10: K
  11: L
  12: M
  13: N
  14: O
  15: P
  16: Q
  17: R
  18: S
  19: T
  20: U
  21: V
  22: W
  23: X
  24: Y
  25: Z
"""

with open("braille_data.yaml", "w") as f:
    f.write(yaml_text)

print(open("braille_data.yaml").read())


path: /content/braille_yolo
train: images/train
val: images/valid
test: images/test

names:
  0: A
  1: B
  2: C
  3: D
  4: E
  5: F
  6: G
  7: H
  8: I
  9: J
  10: K
  11: L
  12: M
  13: N
  14: O
  15: P
  16: Q
  17: R
  18: S
  19: T
  20: U
  21: V
  22: W
  23: X
  24: Y
  25: Z



In [ ]:
# train YOLOv11 for 20 epochs (already svaed from google drive)

from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/braille_yolo_best.pt")

print("Saved YOLO model loaded successfully.")

Saved YOLO model loaded successfully.


In [ ]:
# get the mAP score

best_model = YOLO("runs/detect/braille_runs/yolo11n_20epochs/weights/best.pt")

metrics = best_model.val(
    data="braille_data.yaml",
    split="test",
    imgsz=640
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,587,222 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1011.5±424.5 MB/s, size: 28.9 KB)
val: Scanning /content/braille_yolo/labels/test... 1094 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1094/1094 1.8Kit/s 0.6s
val: New cache created: /content/braille_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 4.3it/s 16.1s
                   all       1094      31166      0.554       0.55      0.539      0.372
                     A        794       2489       0.49      0.605      0.501      0.306
                     B        530        786      0.629      0.339      0.431      0.274
                     C        620       1044       0.71      0.692       0.73      0.499
                     D        690       1569      0.633      0.623    

## ResNet-50 Classification Accuracy

In [ ]:
# generate the crops using the bounding boxes produced by
# the object detection
import json, os, shutil
from pathlib import Path
from PIL import Image

src = Path("braille_dataset")
crop_out = Path("braille_crops")

if crop_out.exists():
    shutil.rmtree(crop_out)

class_names = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
skipped = 0
saved = 0

for split in ["train", "valid", "test"]:
    print(f"Processing {split}...")

    ann_path = src / split / "_annotations.coco.json"

    with open(ann_path, "r") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}

    for cls in class_names:
        (crop_out / split / cls).mkdir(parents=True, exist_ok=True)

    for ann_i, ann in enumerate(coco["annotations"]):
        img_info = images[ann["image_id"]]
        img_path = src / split / img_info["file_name"]

        img = Image.open(img_path).convert("RGB")

        x, y, w, h = ann["bbox"]

        x1 = max(0, int(round(x)))
        y1 = max(0, int(round(y)))
        x2 = min(img.width, int(round(x + w)))
        y2 = min(img.height, int(round(y + h)))

        # Skip invalid/empty crops
        if x2 <= x1 or y2 <= y1:
            skipped += 1
            continue

        crop = img.crop((x1, y1, x2, y2))

        class_id = ann["category_id"] - 1
        class_name = class_names[class_id]

        crop.save(crop_out / split / class_name / f"{img_info['id']}_{ann_i}.jpg")
        saved += 1

print("Done creating cropped classification dataset.")
print("Saved crops:", saved)
print("Skipped invalid crops:", skipped)

Processing train...
Processing valid...
Processing test...
Done creating cropped classification dataset.
Saved crops: 75959
Skipped invalid crops: 0


In [ ]:
# checking the crop counts
import os

for split in ["train", "valid", "test"]:
    total = 0
    for cls in os.listdir(f"braille_crops/{split}"):
        total += len(os.listdir(f"braille_crops/{split}/{cls}"))
    print(split, total)

train 29018
valid 15775
test 31166


In [ ]:
# saves the crops so they do not have to be regenerated

import shutil
import os

crops_folder = "/content/braille_crops"
drive_zip = "/content/drive/MyDrive/braille_crops"

shutil.make_archive(
    drive_zip,
    "zip",
    crops_folder
)

print("Crops saved to Google Drive.")
print("Exists:", os.path.exists(drive_zip + ".zip"))
print("Path:", drive_zip + ".zip")

Crops saved to Google Drive.
Exists: True
Path: /content/drive/MyDrive/braille_crops.zip


In [ ]:
# imports in order to prep for training

import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [ ]:
# create transforms and dataloaders

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("braille_crops/train", transform=train_transform)
valid_dataset = datasets.ImageFolder("braille_crops/valid", transform=eval_transform)
test_dataset = datasets.ImageFolder("braille_crops/test", transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Train: 29018
Valid: 15775
Test: 31166


In [ ]:
# load ResNet-50 and set it up for 26 classes

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 26)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("ResNet-50 ready.")

Using: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]


ResNet-50 ready.


## Accomadating Google Drive Saved Weights

In [ ]:
import torch
import os

# Find your saved ResNet checkpoint in Google Drive
for filename in os.listdir("/content/drive/MyDrive"):
    if "checkpoint" in filename.lower() or "resnet" in filename.lower():
        print(filename)

braille_resnet_checkpoint_epoch_19.pth
resnet50_checkpoint_epoch_30.pth


In [ ]:
checkpoint_path = "/content/drive/MyDrive/resnet50_checkpoint_epoch_30.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

completed_epoch = checkpoint["epoch"]
start_epoch = completed_epoch

print("Loaded checkpoint from epoch:", completed_epoch)
print("Next epoch will be:", start_epoch + 1)

Loaded checkpoint from epoch: 30
Next epoch will be: 31


In [ ]:
# Resume ResNet-50 training from epoch 30 through epoch 50

num_epochs = 50
test_accuracy_readings = {
    30: checkpoint.get("test_accuracy")
}

for epoch in range(start_epoch, num_epochs):
    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    train_loss = train_loss / train_total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            valid_correct += (preds == labels).sum().item()
            valid_total += labels.size(0)

    valid_acc = valid_correct / valid_total
    epoch_number = epoch + 1

    print(
        f"Epoch {epoch_number}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

    # Test and save only at epochs 40 and 50
    if epoch_number in [40, 50]:
        model.eval()

        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, preds = torch.max(outputs, 1)

                test_correct += (preds == labels).sum().item()
                test_total += labels.size(0)

        test_acc = test_correct / test_total
        test_accuracy_readings[epoch_number] = test_acc

        checkpoint_path = (
            f"/content/drive/MyDrive/"
            f"resnet50_checkpoint_epoch_{epoch_number}.pth"
        )

        torch.save(
            {
                "epoch": epoch_number,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "test_accuracy": test_acc
            },
            checkpoint_path
        )

        print(f"Test Accuracy at epoch {epoch_number}: {test_acc:.4f}")
        print(f"Checkpoint saved to: {checkpoint_path}")

Epoch 31/50 | Train Loss: 0.0156 | Train Acc: 0.9950 | Valid Acc: 0.9748
Epoch 32/50 | Train Loss: 0.0132 | Train Acc: 0.9958 | Valid Acc: 0.9698
Epoch 33/50 | Train Loss: 0.0138 | Train Acc: 0.9952 | Valid Acc: 0.9705
Epoch 34/50 | Train Loss: 0.0164 | Train Acc: 0.9954 | Valid Acc: 0.9602
Epoch 35/50 | Train Loss: 0.0099 | Train Acc: 0.9972 | Valid Acc: 0.9666
Epoch 36/50 | Train Loss: 0.0157 | Train Acc: 0.9953 | Valid Acc: 0.9580
Epoch 37/50 | Train Loss: 0.0110 | Train Acc: 0.9964 | Valid Acc: 0.9626
Epoch 38/50 | Train Loss: 0.0091 | Train Acc: 0.9974 | Valid Acc: 0.9660
Epoch 39/50 | Train Loss: 0.0168 | Train Acc: 0.9950 | Valid Acc: 0.9755
Epoch 40/50 | Train Loss: 0.0104 | Train Acc: 0.9969 | Valid Acc: 0.9711
Test Accuracy at epoch 40: 0.9327
Checkpoint saved to: /content/drive/MyDrive/resnet50_checkpoint_epoch_40.pth
Epoch 41/50 | Train Loss: 0.0069 | Train Acc: 0.9977 | Valid Acc: 0.9618
Epoch 42/50 | Train Loss: 0.0145 | Train Acc: 0.9954 | Valid Acc: 0.9685
Epoch 43/50 |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
completed_epoch = 19

checkpoint = {
    "epoch": completed_epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
}

checkpoint_path = (
    f"/content/drive/MyDrive/"
    f"braille_resnet_checkpoint_epoch_{completed_epoch}.pth"
)

torch.save(checkpoint, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)

Checkpoint saved: /content/drive/MyDrive/braille_resnet_checkpoint_epoch_19.pth


In [ ]:
import os

print(os.path.exists(checkpoint_path))
print(os.path.getsize(checkpoint_path) / (1024 ** 2), "MB")

True
270.0920524597168 MB


In [ ]:
# finding yolo to save it for later

import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file in ["best.pt", "last.pt"]:
            print(os.path.join(root, file))

/content/runs/detect/braille_runs/yolo11n_20epochs/weights/last.pt
/content/runs/detect/braille_runs/yolo11n_20epochs/weights/best.pt


In [ ]:
# copying files to YOLO

import shutil
import os

best_source = "/content/runs/detect/braille_runs/yolo11n_20epochs/weights/best.pt"
last_source = "/content/runs/detect/braille_runs/yolo11n_20epochs/weights/last.pt"

best_destination = "/content/drive/MyDrive/braille_yolo_best.pt"
last_destination = "/content/drive/MyDrive/braille_yolo_last.pt"

shutil.copy(best_source, best_destination)
shutil.copy(last_source, last_destination)

print("YOLO weights saved to Google Drive.")
print("Best exists:", os.path.exists(best_destination))
print("Last exists:", os.path.exists(last_destination))

YOLO weights saved to Google Drive.
Best exists: True
Last exists: True


In [ ]:
print("\nFinal classification accuracy readings:")

for epoch, accuracy in test_accuracy_readings.items():
    print(f"{epoch} epochs: {accuracy * 100:.2f}%")

Test Accuracy: 0.8845857665404607
Test Accuracy %: 88.45857665404607
